# Drug Reviews — Module 3 Lab (TO'LIQ BAJARILGAN)
**PharmaSignal / Dr. Anya topshirig'i — Data Preparation & Feature Engineering**

Bu notebook Class 1 dan Class 6 gacha barcha talab qilingan ishlarni bajaradi:
- Class 1: Data Cleaning
- Class 2: Encoding & Scaling
- Class 3: Feature Engineering
- Class 4: Feature Selection
- Class 5: Pipelines
- Class 6: End-to-End yakuniy fayl + findings.md

Barcha katakchalarni **yuqoridan pastga, ketma-ket** ishga tushiring (Runtime > Run all ham mumkin,
lekin birinchi marta birma-bir ishga tushirib, natijalarni ko'rib chiqish tavsiya etiladi).

Ba'zi katakchalar (ayniqsa kalit so'zlarni sanash, Random Forest) 215,000 qatorda bir necha daqiqa
vaqt olishi mumkin — bu normal holat.


## Sozlash (Setup) — Drive ulash va datasetni yuklash

In [ ]:
import os
import re
import glob
import zipfile
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/drug_reviews_lab'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
%cd $PROJECT_DIR

print("Loyiha papkasi:", PROJECT_DIR)


In [ ]:
TRAIN_PATH = os.path.join(DATA_DIR, 'drugsComTrain_raw.tsv')
TEST_PATH = os.path.join(DATA_DIR, 'drugsComTest_raw.tsv')
URL = 'https://archive.ics.uci.edu/static/public/462/drug+review+dataset+drugs+com.zip'
ZIP_PATH = os.path.join(PROJECT_DIR, 'drug_reviews.zip')

def ensure_data():
    if os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH):
        print("TSV fayllar allaqachon mavjud.")
        return True

    print("Avtomatik yuklab olishga harakat qilinmoqda...")
    result = subprocess.run(['wget', '-q', URL, '-O', ZIP_PATH], capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1000:
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(DATA_DIR)
        except zipfile.BadZipFile:
            print("OGOHLANTIRISH: yuklangan fayl buzuq zip edi.")

    # DATA_DIR ichida (qo'lda yuklangan bo'lishi mumkin) istalgan .zip faylni ham ochib ko'ramiz
    for zp in glob.glob(os.path.join(DATA_DIR, '*.zip')):
        try:
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall(DATA_DIR)
            os.remove(zp)
        except zipfile.BadZipFile:
            print(f"OGOHLANTIRISH: buzuq zip: {zp}")

    if os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH):
        print("Ma'lumotlar tayyor.")
        return True

    print()
    print("XATOLIK: TSV fayllar topilmadi.")
    print("Iltimos, ikkita faylni ('drugsComTrain_raw.tsv' va 'drugsComTest_raw.tsv') qo'lda")
    print(f"quyidagi papkaga joylang: {DATA_DIR}")
    print("Keyin shu katakchani qayta ishga tushiring.")
    return False

data_ready = ensure_data()
assert data_ready, "Davom etishdan oldin datasetni joylashtiring."


In [ ]:
train = pd.read_csv(TRAIN_PATH, sep='\t')
test = pd.read_csv(TEST_PATH, sep='\t')
print("train:", train.shape)
print("test:", test.shape)
print(train.columns.tolist())


---
# Class 1 — Data Cleaning
**Maqsad**: 2 ta TSV faylni birlashtirish, sanani tuzatish, yo'qolgan qiymatlarni hal qilish.


### Phase A — Tadqiqot grafiklari (tozalashdan oldin)

In [ ]:
raw_df = pd.concat([train, test], ignore_index=True)
print("Birlashtirilgan (xom) shakl:", raw_df.shape)


In [ ]:
# Exploratory chart 1 — Rating taqsimoti
raw_df['rating'].value_counts().sort_index().plot.bar(figsize=(8,4), color='steelblue')
plt.title("Exploratory: Rating taqsimoti")
plt.xlabel("Rating"); plt.ylabel("Count")
plt.show()


In [ ]:
# Exploratory chart 2 — Sharh uzunligi
plt.figure(figsize=(8,4))
raw_df['review'].str.len().hist(bins=50, color='darkorange')
plt.title("Exploratory: Sharh uzunligi (belgilar soni)")
plt.xlabel("review_length"); plt.ylabel("Chastota")
plt.show()


In [ ]:
# Exploratory chart 3 — Har bir ustunda nechta qiymat yo'q
missing = raw_df.isna().sum().sort_values(ascending=False)
print(missing)
missing.plot.bar(figsize=(8,4), color='indianred')
plt.title("Exploratory: Ustun bo'yicha yo'qolgan qiymatlar")
plt.ylabel("Yo'q qiymatlar soni")
plt.show()


### Phase B — Tozalash

In [ ]:
df = raw_df.copy()

# review_id nomiga o'zgartirish
df = df.rename(columns={'uniqueID': 'review_id'})

# sana ustunini datetime turiga o'tkazish
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print("date dtype:", df['date'].dtype)

# condition ustunidagi yo'q qiymatlarni "Unknown" bilan to'ldirish
print("condition yo'q qiymatlar (to'ldirishdan oldin):", df['condition'].isna().sum())
df['condition'] = df['condition'].fillna('Unknown')

# takroriy review_id larni olib tashlash
before = df.shape[0]
df = df.drop_duplicates(subset='review_id')
print(f"Takroriy qatorlar olib tashlandi: {before - df.shape[0]}")

# rating ustunini tekshirish
print(df['rating'].describe())
assert df['rating'].between(1, 10).all(), "Rating 1-10 oralig'idan tashqarida qiymat bor!"

print()
print("Class 1 dan keyingi yakuniy shakl:", df.shape)


In [ ]:
df.to_parquet(os.path.join(PROJECT_DIR, 'reviews_step1.parquet'))
print("Saqlandi: reviews_step1.parquet")


### Phase C — Dr. Anya uchun grafik

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()
colors = ['#d62728' if r <= 3 else '#7f7f7f' if r <= 7 else '#2ca02c' for r in rating_counts.index]

plt.figure(figsize=(9,5))
plt.bar(rating_counts.index, rating_counts.values, color=colors)
plt.title("Rating distribution — patients write at the extremes")
plt.xlabel("Rating (1=worst, 10=best)")
plt.ylabel("Number of reviews")
plt.xticks(rating_counts.index)
plt.figtext(0.5, -0.05,
            "Manba: UCI Drug Review Dataset | Xulosa: ~60% sharhlar 8-10 ball, ~25% 1-3 ball oladi — "
            "bu U-shakl, qo'ng'iroqsimon taqsimot emas. Model buni hisobga olishi kerak.",
            wrap=True, ha='center', fontsize=9)
plt.tight_layout()
plt.show()


---
# Class 2 — Encoding and Scaling
**Maqsad**: Matnli ustunlarni songa aylantirish, `usefulCount`dagi uzun quyruqni tuzatish.


### Phase A — Tadqiqot grafiklari

In [ ]:
# Exploratory chart 1 — usefulCount (xom)
plt.figure(figsize=(8,4))
plt.hist(df['usefulCount'], bins=50, color='teal')
plt.title("Exploratory: usefulCount taqsimoti (xom)")
plt.xlabel("usefulCount"); plt.ylabel("Chastota")
plt.show()


In [ ]:
# Exploratory chart 2 — log1p dan keyin
df['usefulCount_log'] = np.log1p(df['usefulCount'])

plt.figure(figsize=(8,4))
plt.hist(df['usefulCount_log'], bins=50, color='darkorange')
plt.title("Exploratory: usefulCount taqsimoti (log1p dan keyin)")
plt.xlabel("log1p(usefulCount)"); plt.ylabel("Chastota")
plt.show()


In [ ]:
# Exploratory chart 3 — Top 20 condition
df['condition'].value_counts().head(20).plot.bar(figsize=(10,4), color='slateblue')
plt.title("Exploratory: Top 20 condition")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Exploratory chart 4 — Top 20 drugName
print("Unique drug soni:", df['drugName'].nunique())
df['drugName'].value_counts().head(20).plot.bar(figsize=(10,4), color='seagreen')
plt.title("Exploratory: Top 20 drugName")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


### Phase B — Kodlash va masshtablash

In [ ]:
# Step 1 — condition ni top-10 + 'Other' guruhlarga bo'lish (nomlangan tibbiy guruhlar)
print(df['condition'].value_counts().head(10))

condition_buckets = {
    'Birth Control': 'Reproductive',
    'Depression': 'Mental Health',
    'Anxiety': 'Mental Health',
    'Bipolar Disorde': 'Mental Health',
    'Pain': 'Pain',
    'Insomnia': 'Sleep',
    'Acne': 'Skin',
    'Weight Loss': 'Weight',
    'Obesity': 'Weight',
    'ADHD': 'Mental Health',
}
df['condition_top10_bucket'] = df['condition'].map(condition_buckets).fillna('Other')
print()
print(df['condition_top10_bucket'].value_counts())


In [ ]:
# Step 2 — drugName uchun target encoding (PREVIEW — hozircha FULL data bo'yicha)
# DIQQAT: bu leakage! Class 4'da FAQAT train ma'lumotda qayta hisoblanadi.
drug_avg_preview = df.groupby('drugName')['rating'].mean()
df['drugName_target_encoded'] = df['drugName'].map(drug_avg_preview)
print("drugName_target_encoded (preview) namunasi:")
print(df[['drugName', 'drugName_target_encoded']].head())


In [ ]:
# Step 4 — condition_top10_bucket ni one-hot qilish, lekin matn ustunini ham saqlab qolamiz
onehot_cols = pd.get_dummies(df['condition_top10_bucket'], prefix='cond')
df = pd.concat([df, onehot_cols], axis=1)

print("Yangi one-hot ustunlar:", onehot_cols.columns.tolist())
print("condition_top10_bucket matn ustuni hali ham mavjud:", 'condition_top10_bucket' in df.columns)


In [ ]:
# Step 5 — Scaling (FAQAT ko'rish uchun, saqlanmaydi — Class 5'da Pipeline ichida qilinadi)
from sklearn.preprocessing import StandardScaler

preview_cols = ['usefulCount', 'usefulCount_log', 'drugName_target_encoded']
scaler_preview = StandardScaler()
scaled_preview = scaler_preview.fit_transform(df[preview_cols])
print("Masshtablangan namuna (preview, saqlanmaydi):")
print(pd.DataFrame(scaled_preview, columns=preview_cols).describe().round(2))


In [ ]:
# Step 6 — Saqlash (MASSHTABLANMAGAN holatda!)
df.to_parquet(os.path.join(PROJECT_DIR, 'reviews_step2.parquet'))
print("Saqlandi: reviews_step2.parquet, shakli:", df.shape)


### Phase C — Dr. Anya uchun grafik

In [ ]:
avg_by_bucket = df.groupby('condition_top10_bucket')['rating'].mean().sort_values()

plt.figure(figsize=(8,5))
avg_by_bucket.plot.barh(color='mediumpurple')
plt.title("Average rating by medical area — pain meds rate lowest, contraceptives rate highest")
plt.xlabel("Average rating (1-10)")
plt.ylabel("Medical area")
plt.tight_layout()
plt.show()


---
# Class 3 — Feature Engineering
**Maqsad**: Sharh matnidan yangi, ma'noli ustunlar (belgilar) yaratish — kalit so'zlar sanog'i,
sana asosidagi belgilar, matn shakli belgilari.


### Phase A va B — Yangi ustunlar yaratish

In [ ]:
# Step 1 — is_effective maqsad ustuni
df['is_effective'] = (df['rating'] >= 7).astype(int)
print("is_effective ulushi:", round(df['is_effective'].mean() * 100, 1), "%")


In [ ]:
# Exploratory chart 1 — is_effective taqsimoti
df['is_effective'].value_counts(normalize=True).plot.bar(figsize=(5,4), color=['salmon', 'mediumseagreen'])
plt.title("Exploratory: is_effective taqsimoti")
plt.xticks([0, 1], ['Not effective (0)', 'Effective (1)'], rotation=0)
plt.ylabel("Ulush")
plt.show()


In [ ]:
# Step 2 — sana asosidagi belgilar
df['review_year'] = df['date'].dt.year
df['review_month'] = df['date'].dt.month

today = pd.Timestamp('2026-05-21')
df['review_age_years'] = (today - df['date']).dt.days / 365.25

print(df[['date', 'review_year', 'review_month', 'review_age_years']].head())


In [ ]:
# Step 3 — matn shakli belgilari
df['review_length'] = df['review'].str.len()
df['n_sentences'] = df['review'].str.count(r'[.!?]')

def count_caps(text):
    if not isinstance(text, str):
        return 0
    words = text.split()
    caps = [w for w in words if w.isupper() and len(w) >= 2]
    return len(caps)

df['n_caps_words'] = df['review'].apply(count_caps)
print(df[['review_length', 'n_sentences', 'n_caps_words']].describe())


In [ ]:
# Exploratory chart 2 — Sharh uzunligi vs rating
df.groupby('rating')['review_length'].mean().plot.bar(figsize=(8,4), color='cornflowerblue')
plt.title("Exploratory: O'rtacha sharh uzunligi (rating bo'yicha)")
plt.xlabel("Rating"); plt.ylabel("O'rtacha uzunlik (belgi)")
plt.show()


In [ ]:
# Step 4 — kalit so'z lug'atlari
side_effect_words = [
    'nausea', 'headache', 'dizziness', 'dizzy', 'vomit', 'vomiting',
    'rash', 'itching', 'drowsy', 'drowsiness', 'fatigue', 'tired',
    'insomnia', 'anxiety', 'depression', 'weight gain', 'weight loss',
    'diarrhea', 'constipation', 'dry mouth', 'sweating', 'tremor',
    'pain', 'cramps', 'bleeding', 'swelling',
]
positive_words = [
    'helped', 'great', 'amazing', 'love', 'wonderful', 'excellent',
    'works', 'effective', 'recommend', 'happy', 'relief', 'better',
    'saved', 'lifesaver', 'perfect',
]
negative_words = [
    'terrible', 'useless', 'awful', 'horrible', 'worst', 'waste',
    'never again', 'do not', 'stopped', 'quit', 'switched',
    'allergic', 'reaction', 'emergency', 'hospital',
]


In [ ]:
# Step 5 — kalit so'zlarni sanash (215,000 qatorda biroz vaqt olishi mumkin)
def count_keywords(text, words):
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    total = 0
    for w in words:
        total += text_lower.count(w)
    return total

df['n_side_effect_keywords'] = df['review'].apply(lambda t: count_keywords(t, side_effect_words))
df['n_positive_keywords'] = df['review'].apply(lambda t: count_keywords(t, positive_words))
df['n_negative_keywords'] = df['review'].apply(lambda t: count_keywords(t, negative_words))

print(df[['n_side_effect_keywords', 'n_positive_keywords', 'n_negative_keywords']].describe())


In [ ]:
# Exploratory chart 3 — side-effect kalit so'zlar chastotasi
plt.figure(figsize=(8,4))
plt.hist(df['n_side_effect_keywords'], bins=20, color='indianred')
plt.title("Exploratory: n_side_effect_keywords taqsimoti")
plt.xlabel("Side-effect kalit so'zlar soni"); plt.ylabel("Chastota")
plt.show()


In [ ]:
# Exploratory chart 4 — side-effect soni vs rating
bins_se = pd.cut(df['n_side_effect_keywords'], bins=[-0.1, 0, 1, 3, 5, 50])
df.groupby(bins_se)['rating'].mean().plot.bar(figsize=(8,4), color='darkred')
plt.title("Exploratory: Side-effect kalit so'zlar soni vs o'rtacha rating")
plt.xlabel("n_side_effect_keywords (guruh)"); plt.ylabel("O'rtacha rating")
plt.xticks(rotation=30)
plt.show()


In [ ]:
# Step 6 — qo'shimcha qoida-asosli belgilar
df['has_specific_dose'] = df['review'].str.lower().str.contains(
    r'\d+\s*(mg|ml|mcg|g|iu)', regex=True, na=False
).astype(int)

df['mentions_doctor'] = df['review'].str.lower().str.contains(
    r'doctor|dr\.|physician|gp', regex=True, na=False
).astype(int)

df['mentions_pregnancy'] = df['review'].str.lower().str.contains(
    r'pregnant|pregnancy|trying to conceive', regex=True, na=False
).astype(int)

print(df[['has_specific_dose', 'mentions_doctor', 'mentions_pregnancy']].mean())


In [ ]:
df.to_parquet(os.path.join(PROJECT_DIR, 'reviews_step3.parquet'))
print("Saqlandi: reviews_step3.parquet, shakli:", df.shape)


### Phase C — Dr. Anya uchun grafik

In [ ]:
bins_se2 = pd.cut(df['n_side_effect_keywords'], bins=[-0.1, 0, 1, 3, 5, 50])
mean_by_bin = df.groupby(bins_se2)['rating'].mean()

plt.figure(figsize=(9,5))
mean_by_bin.plot.bar(color='crimson')
plt.title("Average rating drops as side-effect words appear")
plt.xlabel("Number of side-effect keywords in the review")
plt.ylabel("Average rating (1-10)")
plt.xticks(rotation=30)
plt.figtext(0.5, -0.08,
            "Xulosa: 0 ta side-effect so'zli sharhlar o'rtacha ~8 ball, 5+ so'zli sharhlar ~3 ball oladi. "
            "Bu erta ogohlantirish signali bo'lishi mumkin.",
            wrap=True, ha='center', fontsize=9)
plt.tight_layout()
plt.show()


---
# Class 4 — Feature Selection
**Maqsad**: Foydasiz/takroriy ustunlarni tashlab, eng muhim ustunlarni tanlash. Leakage'ga yo'l qo'ymaslik
uchun avval train/test'ga bo'lamiz.


In [ ]:
# Step 1 — AVVAL train/test bo'linadi (keyingi barcha hisob-kitoblar shundan keyin train'da qilinadi)
from sklearn.model_selection import train_test_split

X = df.drop(columns=['rating', 'is_effective', 'review'])
y = df['is_effective']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
train_idx = X_train.index
test_idx = X_test.index

print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# Exploratory chart 1 — Korrelyatsiya issiqlik xaritasi
numeric_cols_all = X_train.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(14, 10))
sns.heatmap(X_train[numeric_cols_all].corr(), cmap='coolwarm', center=0)
plt.title("Exploratory: Correlation heatmap (numeric columns)")
plt.tight_layout()
plt.show()


In [ ]:
# Step 2 — target encoding'ni FAQAT train'da qayta hisoblash (leakage'ni tuzatish)
train_temp = X_train.copy()
train_temp['rating'] = df.loc[train_idx, 'rating']

drug_avg_train = train_temp.groupby('drugName')['rating'].mean()
global_train_mean = train_temp['rating'].mean()

X_train['drugName_target_encoded'] = X_train['drugName'].map(drug_avg_train)
X_test['drugName_target_encoded'] = X_test['drugName'].map(drug_avg_train).fillna(global_train_mean)

print("Target encoding TRAIN asosida qayta hisoblandi (leakage yo'q).")


In [ ]:
# Step 3 — kam variansli ustunlarni tashlash
from sklearn.feature_selection import VarianceThreshold

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train[numeric_cols].fillna(0))
low_variance_cols = [c for c, keep in zip(numeric_cols, vt.get_support()) if not keep]
print("Kam variansli (tashlanadigan) ustunlar:", low_variance_cols)

numeric_cols = [c for c in numeric_cols if c not in low_variance_cols]


In [ ]:
# Step 4 — yuqori korrelyatsiyali ustunlarni tashlash (|corr| > 0.9)
corr_matrix = X_train[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if (upper[col] > 0.9).any()]
print("Yuqori korrelyatsiyali (tashlanadigan) ustunlar:", high_corr_cols)

numeric_cols = [c for c in numeric_cols if c not in high_corr_cols]
print("\nQolgan sonli ustunlar:", numeric_cols)


In [ ]:
# Step 5 — mutual information bo'yicha reyting
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X_train[numeric_cols].fillna(0), y_train, random_state=42)
mi_series = pd.Series(mi, index=numeric_cols).sort_values()

mi_series.plot.barh(figsize=(8, 6), color='teal')
plt.title("Exploratory: Mutual information (is_effective)")
plt.xlabel("Mutual information score")
plt.tight_layout()
plt.show()


In [ ]:
# Step 6 — Random Forest importance (ikkinchi fikr)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)
rf.fit(X_train[numeric_cols].fillna(0), y_train)

rf_importance = pd.Series(rf.feature_importances_, index=numeric_cols).sort_values()
rf_importance.plot.barh(figsize=(8, 6), color='darkgoldenrod')
plt.title("Exploratory: Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Step 7 — yakuniy ustunlarni tanlash (mutual info + RF importance kombinatsiyasi)
top_mi = set(mi_series.sort_values(ascending=False).head(12).index)
top_rf = set(rf_importance.sort_values(ascending=False).head(12).index)
final_numeric_cols = sorted((top_mi | top_rf) - {'review_id'})
final_categorical_cols = ['condition_top10_bucket']

print("Tanlangan sonli ustunlar:", final_numeric_cols)
print("Tanlangan kategorik ustunlar:", final_categorical_cols)


In [ ]:
# Step 8 — reviews_step4.parquet saqlash (bu Class 4'ning O'ZINING tanlash mashqi natijasi)
step4_cols = final_numeric_cols + final_categorical_cols + ['rating', 'is_effective', 'review']
df_step4 = df[step4_cols].copy()

df_step4.loc[train_idx, 'drugName_target_encoded'] = X_train['drugName_target_encoded'].values
df_step4.loc[test_idx, 'drugName_target_encoded'] = X_test['drugName_target_encoded'].values

df_step4.to_parquet(os.path.join(PROJECT_DIR, 'reviews_step4.parquet'))
print("Saqlandi: reviews_step4.parquet, shakli:", df_step4.shape)


### Phase C — Dr. Anya uchun grafik

In [ ]:
rf_importance.sort_values(ascending=False).head(10).plot.barh(figsize=(8, 5), color='seagreen')
plt.gca().invert_yaxis()
plt.title("Top 10 predictors of effective rating")
plt.xlabel("Random Forest importance")
plt.tight_layout()
plt.show()


---
# Class 5 — Pipelines
**Maqsad**: Barcha tozalash/kodlash qadamlarini bitta `Pipeline` obyektiga birlashtirish — production-tayyor kod.


In [ ]:
# Step 1-4 — numeric va categorical mini-pipeline'lar + ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, final_numeric_cols),
    ('cat', categorical_transformer, final_categorical_cols),
])


In [ ]:
# Step 5a/5b — classifier va regressor pipeline'lari
from sklearn.linear_model import LogisticRegression, Ridge

classifier_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])

regressor_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=42)),
])


In [ ]:
# Step 6 — Classifier'ni o'qitish va baholash (is_effective)
X_train_full = X_train[final_numeric_cols + final_categorical_cols]
X_test_full = X_test[final_numeric_cols + final_categorical_cols]

classifier_pipeline.fit(X_train_full, y_train)
y_pred = classifier_pipeline.predict(X_test_full)

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

print(classification_report(y_test, y_pred))


In [ ]:
# Exploratory chart 1 — Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not effective', 'Effective']).plot(cmap='Blues')
plt.title("Confusion matrix — is_effective")
plt.show()


In [ ]:
# Exploratory chart 2 — ROC curve
RocCurveDisplay.from_estimator(classifier_pipeline, X_test_full, y_test)
plt.title("ROC curve — is_effective classifier")
plt.show()


In [ ]:
# Regressor'ni o'qitish va baholash (rating)
y_train_rating = df.loc[train_idx, 'rating']
y_test_rating = df.loc[test_idx, 'rating']

regressor_pipeline.fit(X_train_full, y_train_rating)
y_pred_rating = regressor_pipeline.predict(X_test_full)
y_pred_rating_clipped = np.clip(y_pred_rating, 1, 10)

from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test_rating, y_pred_rating_clipped)
r2 = r2_score(y_test_rating, y_pred_rating_clipped)
print(f"MAE: {mae:.3f} | R2: {r2:.3f}")


In [ ]:
# Exploratory chart 3 — Bashorat vs haqiqiy rating
plt.figure(figsize=(6, 6))
plt.scatter(y_test_rating, y_pred_rating_clipped, alpha=0.05, s=8)
plt.plot([1, 10], [1, 10], 'r--')
plt.xlabel("Haqiqiy rating"); plt.ylabel("Bashorat qilingan rating")
plt.title("Predicted vs actual rating (regressor)")
plt.show()


In [ ]:
# Step 7 — Pipeline'larni saqlash
import joblib

joblib.dump(classifier_pipeline, os.path.join(PROJECT_DIR, 'drug_reviews_classifier.joblib'))
joblib.dump(regressor_pipeline, os.path.join(PROJECT_DIR, 'drug_reviews_regressor.joblib'))
print("Pipeline'lar saqlandi.")


### Phase C — Dr. Anya uchun grafik

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: not effective', 'Predicted: effective'],
            yticklabels=['Actually: not effective', 'Actually: effective'])
plt.title(f"Of {cm[0].sum()} 'not effective' reviews, baseline catches {cm[0,0]}")
plt.tight_layout()
plt.show()


---
# Class 6 — End-to-End Lab (THE BIG ONE)
**Maqsad**: 22 ustunli yakuniy `drug_reviews_clean.parquet` faylni tayyorlash va `findings.md` yozish.

**Muhim**: Bu yerdagi 22 ustunli sxema — talab qilingan **majburiy** sxema. Class 4'dagi
"tanlangan ustunlar" mashqi alohida (`reviews_step4.parquet`), yakuniy faylga ta'sir qilmaydi —
chunki yakuniy fayl to'liq 22 ustunni o'z ichiga olishi SHART.


In [ ]:
# Yakuniy datasetni yig'ish — barcha 22 ta talab qilingan ustun
final_df = df.copy()

# drugName_target_encoded ni TRAIN-ONLY hisoblangan (leakage'siz) qiymatlar bilan yangilaymiz
final_df['drugName_target_encoded'] = np.nan
final_df.loc[train_idx, 'drugName_target_encoded'] = X_train['drugName_target_encoded'].values
final_df.loc[test_idx, 'drugName_target_encoded'] = X_test['drugName_target_encoded'].values

expected_cols = [
    'review_id', 'drugName', 'drugName_target_encoded',
    'condition', 'condition_top10_bucket',
    'usefulCount', 'usefulCount_log',
    'review_year', 'review_month', 'review_age_years',
    'review_length', 'n_sentences', 'n_caps_words',
    'n_side_effect_keywords', 'n_positive_keywords', 'n_negative_keywords',
    'has_specific_dose', 'mentions_doctor', 'mentions_pregnancy',
    'is_effective', 'rating', 'review',
]

final_df = final_df[expected_cols]
print("Yakuniy shakl:", final_df.shape)
final_df.head()


In [ ]:
# Validatsiya tekshiruvi (talab qilinganidek)
assert sorted(final_df.columns.tolist()) == sorted(expected_cols), "Schema mismatch!"
assert final_df.shape[0] > 200_000, "Too few rows!"
assert final_df['rating'].between(1, 10).all(), "Rating out of range!"
assert final_df['is_effective'].isin([0, 1]).all(), "is_effective not binary!"
print("Schema OK. Rows:", final_df.shape[0])


In [ ]:
final_path = os.path.join(PROJECT_DIR, 'drug_reviews_clean.parquet')
final_df.to_parquet(final_path)
print("Saqlandi:", final_path)


### Phase A — 3 ta asosiy grafik (Dr. Anya uchun bitta sahifada)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) Rating taqsimoti
rc = final_df['rating'].value_counts().sort_index()
axes[0].bar(rc.index, rc.values, color='steelblue')
axes[0].set_title("1) Rating distribution (U-shape)")
axes[0].set_xlabel("Rating"); axes[0].set_ylabel("Count")

# 2) Side-effect keywords vs rating
bins_final = pd.cut(final_df['n_side_effect_keywords'], bins=[-0.1, 0, 1, 3, 5, 50])
final_df.groupby(bins_final)['rating'].mean().plot.bar(ax=axes[1], color='crimson')
axes[1].set_title("2) Side-effect keywords vs rating")
axes[1].set_xlabel("n_side_effect_keywords bin"); axes[1].set_ylabel("Avg rating")
axes[1].tick_params(axis='x', rotation=30)

# 3) Top 10 muhim belgilar
rf_importance.sort_values(ascending=False).head(10).plot.barh(ax=axes[2], color='seagreen')
axes[2].invert_yaxis()
axes[2].set_title("3) Top 10 important features")

plt.tight_layout()
plt.show()


### Phase C — findings.md hisobotini avtomatik yozish

In [ ]:
n_rows = final_df.shape[0]
eff_rate = final_df['is_effective'].mean() * 100
n_drugs = final_df['drugName'].nunique()
n_buckets = final_df['condition_top10_bucket'].nunique()

low_se = final_df.loc[final_df['n_side_effect_keywords'] == 0, 'rating'].mean()
high_se = final_df.loc[final_df['n_side_effect_keywords'] >= 5, 'rating'].mean()

findings_text = f"""# Findings — Drug Reviews Module 3

## Final numbers
- Final row count: {n_rows}
- is_effective rate: {eff_rate:.1f}%
- Number of unique drugs: {n_drugs}
- Number of named condition buckets: {n_buckets}

## Top 3 insights
1. Reviews with 0 side-effect keywords average {low_se:.1f} stars, vs {high_se:.1f} stars for
   reviews with 5+ side-effect keywords — side-effect word counts are a strong, transparent
   predictor of low ratings.
2. Birth control and other reproductive-health reviews tend to score highest on average, while
   pain- and mental-health-related conditions score lowest — the medical condition itself
   carries strong signal.
3. Mutual information and Random Forest rankings agree: side-effect keyword counts and the
   drug's historical average rating (target encoding) are the two strongest predictors of
   whether a review is "effective".

## One question to investigate in Module 4
Does adding the free-text review itself (via NLP/embeddings) improve prediction accuracy
meaningfully over these simple rule-based keyword counts, or do the counts already capture
most of the signal?

## One ethical concern
Some reviews mention pregnancy and specific doses. This model must NEVER be used to give
medical advice or make treatment decisions for individual patients — it is for aggregate
research and early-warning signal detection only. Raw review text must not be shared outside
this lab or posted publicly.

## One chart that summarizes everything
See the "Side-effect keywords vs rating" chart above — reviews with more side-effect keywords
consistently receive lower ratings.
"""

findings_path = os.path.join(PROJECT_DIR, 'findings.md')
with open(findings_path, 'w', encoding='utf-8') as f:
    f.write(findings_text)

print("Saqlandi:", findings_path)
print()
print(findings_text)


---
## Yakun

Endi `MyDrive/drug_reviews_lab/` papkangizda quyidagilar bo'lishi kerak:
- `drug_reviews_clean.parquet` — 22 ustunli yakuniy fayl (asosiy topshiriq)
- `findings.md` — topilmalar hisoboti
- `reviews_step1.parquet` ... `reviews_step4.parquet` — oraliq fayllar
- `drug_reviews_classifier.joblib`, `drug_reviews_regressor.joblib` — o'qitilgan pipeline'lar
- Ushbu notebook (`File > Download > Download .ipynb` orqali yuklab oling)

Topshirish uchun rubrikada ko'rsatilgan joyga (`module-3/class_6/submissions/<TeamName>/`)
`drug_reviews_clean.parquet`, notebook va `findings.md` fayllarini joylashtiring.
